In [1]:
# Parameters
# ver = '1.2.1.4.2.4'

# Training algorithm for O2 gapfill
    - Needs to enter a 6 digit input parameter as follows : 
    - First digit = Algorithm type (1=RF, 2=NN)
    - Second digit = Data Source (1=Ship only, 2=Ship+Argo)
    - Third digit = Ocean basin (1=Atlantic, 2=Pacific, 3=Indian, 4=Southern, 5=Arctic)
    - Fourth digit = T/S data source (1=EN4,2=ORAS,3=SODA, 6=IPSL-CM6A-LR, 7=GFDL-ESM4)
    - Fifth digit = predictor variable set (1=default, 2=cos/sin_month)
    - Sixth digit = hyperparameter set (1=default, 2=preset hyperparameters)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import sklearn as skl
import gsw
import cartopy.crs as ccrs
from scipy.interpolate import interp1d
import os
import warnings
warnings.filterwarnings('ignore')

In [17]:
# version information
ver = '1.2.3.6.2.4'
date1='03282025' # Set this for saving today's date. Usually date1=today's date
date2='03282025' # Set alternative date for re-running previous results
rerun = False    # indicate again whether you are re-running previous results
model = 'IPSL-CM6A-LR'

### display selection

In [18]:
selection = ver.split('.')
basin = ['Atlantic','Pacific','Indian','Southern','Arctic']
#
if selection[0] == '1':
    print('Random Forst algorithm will be used.')
    alg = 'RF'
elif selection[0] == '2':
    print('Neural Network algorithm will be used.')
    alg = 'NN'
else:
    print('error - incorrect algorithm type')
#
if selection[1] == '1':
    print('Ship-based O2 data will be used. Year_end = 2011')
    endyear=2011
elif selection[1] == '2':
    print('Ship-based and Argo-O2 data will be used. Year_end = 2021')
    endyear=2021
else:
    print('error - incorrect input data type')
#
if selection[2] == '1':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '2':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '3':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '4':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '5':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
else:
    print('error - incorrect O2 data type')
#
if selection[3] == '1':
    print('EN4 dataset will be used for T/S input. ')
elif selection[3] == '3':
    print('SODA3.4.2 dataset will be used for T/S input. ')
elif selection[3] == '4':
    print('ORAS5 dataset will be used for T/S input. ')
elif selection[3] == '5':
    print('CanESM5 dataset will be used for T/S input. ')
elif selection[3] == '6':
    print('IPSL-CM6A-LR dataset will be used for T/S input. ')
elif selection[3] == '7':
    print('GFDL-ESM4 dataset will be used for T/S input. ')
else:
    print('error - incorrect T/S data type')
#
if selection[4] == '1':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, month')
elif selection[4] == '2':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)')
elif selection[4] == '3':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma')
elif selection[4] == '4':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma, N2')
else:
    print('error - incorrect predictor variable type')
#
if selection[5] == '1':
    print('Hyperparameter set is optimized via K-fold CV')
elif selection[5] == '2':
    print('A pre-set hyperparameter set is used')
elif selection[5] == '4':
    print('New K-fold cross validation')
else:
    print('error - incorrect hyperparameter type')

Random Forst algorithm will be used.
Ship-based and Argo-O2 data will be used. Year_end = 2021
Indian Ocean will be mapped
IPSL-CM6A-LR dataset will be used for T/S input. 
Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)
New K-fold cross validation


In [19]:
# Define the input and output folders
#
# os.system('echo $USER > userid')
# usrid=np.genfromtxt('userid',dtype='<U8')
# os.system('rm userid')
usrid = 'qzhang459'
#
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/CMIP6/{model}/ML4O2_temp')
dirout=f'/glade/derecho/scratch/qzhang459/CMIP6/{model}/'
diro = '/glade/derecho/scratch/'+str(usrid)+'/WOD18_OSDCTD/'
dirf = f'/glade/campaign/univ/ugit0034/cmip6/{model}/TS/'
dirin = '/glade/campaign/univ/ugit0034/WOD18_OSDCTD/'
fargo = '/glade/campaign/univ/ugit0034/bgcargo/o2_Global_ARGO_Type12_47lev.nc'
fosd='_1x1bin_osd_'
fctd='_1x1bin_ctd_'
fmer='_1x1bin_osdctd_'
var=['o2','TSN2']
os.system('mkdir -p '+diro)
os.system('mkdir -p '+diro+'temp')

0

### Preprocessing the data

In [20]:
# obtain vertical grid
ds=xr.open_dataset(dirin+var[0]+fmer+str(1980)+'.nc')
Z=ds.depth.to_numpy()
Nz=np.size(Z)

In [21]:
# select analysis period
# do not change the start year from 1965 (this is when Carpenter 1965 established modern Winkler method)
yrs=np.arange(1980,endyear,1)

In [22]:
# basin-specific input data loading
dsm=xr.open_dataset('/glade/campaign/univ/ugit0034/wod18/basin_mask_01.nc')
#
if selection[2] == '1':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='atlantic'
elif selection[2] == '2':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='pacific'
elif selection[2] == '3':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='indian'
elif selection[2] == '4':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='southern'
elif selection[2] == '5':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='arctic'
else:
    print('error - incorrect O2 data type')
#
dinput = f'/glade/campaign/univ/ugit0034/ML4O2/CMIP6/{model}/'
if selection[1]=='2':
    doa1 = np.load(dinput+f'o20_{bname0}_1x1_47lev.npy')
    dta1 = np.load(dinput+f't0_{bname0}_1x1_47lev.npy')
    dsa1 = np.load(dinput+f's0_{bname0}_1x1_47lev.npy')
    xx1 = np.load(dinput+f'lon0_{bname0}_1x1_47lev.npy')
    yy1 = np.load(dinput+f'lat0_{bname0}_1x1_47lev.npy')
    zz1 = np.load(dinput+f'depth0_{bname0}_1x1_47lev.npy')
    tt1 = np.load(dinput+f'time0_{bname0}_1x1_47lev.npy')
    tc1 = np.load(dinput+f'month0_{bname0}_1x1_47lev.npy')
    dsga1 = np.load(dinput+f'sigma0_{bname0}_1x1_47lev.npy')
    # dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ORAS4/SODAINPUT/N20_{bname0}_1x1_47lev.npy')
elif selection[1]=='1':
    doa1 = np.load(dinput+f'o20_{bname0}_1x1_47lev_ship.npy')
    dta1 = np.load(dinput+f't0_{bname0}_1x1_47lev_ship.npy')
    dsa1 = np.load(dinput+f's0_{bname0}_1x1_47lev_ship.npy')
    xx1 = np.load(dinput+f'lon0_{bname0}_1x1_47lev_ship.npy')
    yy1 = np.load(dinput+f'lat0_{bname0}_1x1_47lev_ship.npy')
    zz1 = np.load(dinput+f'depth0_{bname0}_1x1_47lev_ship.npy')
    tt1 = np.load(dinput+f'time0_{bname0}_1x1_47lev_ship.npy')
    tc1 = np.load(dinput+f'month0_{bname0}_1x1_47lev_ship.npy')
    dsga1 = np.load(dinput+f'sigma0_{bname0}_1x1_47lev_ship.npy')
    # dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/N20_{bname0}_1x1_47lev_ship.npy')

Indian Ocean will be mapped


In [23]:
Nsample = np.size(doa1)
print(Nsample)

694799


### This is where we choose what variables to include

In [24]:
# generate data matrix and standardize it
if selection[4] == '1':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, tc1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, month')
elif selection[4] == '2':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)')
elif selection[4] == '3':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12), dsga1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma')
elif selection[4] == '4':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12), dsga1, dn2a1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma, N2')
else:
    print('error - incorrect predictor variable type')    
#X = np.array([dsa1, dta1, xx1, yy1, tt1, tc1])
#
y = doa1
#
Xm = np.mean(X,axis=1)
Xstd = np.std(X,axis=1)
#
N=np.size(y)
# normalize x and y
Xa = (X.T - Xm)/Xstd
ym = np.mean(y)
ystd = np.std(y)
ya = (y-ym)/ystd
#
np.savez(dirout+f'ML_params_v{ver}.npz',Xm=Xm,Xstd=Xstd,ym=ym,ystd=ystd)

Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)
